# **Hill Country Grocer Revenue Prediction**: Supervised Regression Ensemble
----------------------------------------


## **Value Proposition**

In order to support weekly inventory and pricing decisions across a four-store regional grocer, a tabular regression ensemble was developed that predicts per-product weekly revenue with {{PASS_2: primary_r2}} R-squared and {{PASS_2: primary_mape}} MAPE on the held-out test set. With approximately 8,880 product-store-week observations across four Hill Country Grocer locations in South-Central Texas, accurate revenue prediction reduces overstock waste on perishables and prevents lost sales on stockouts, directly affecting margin in a category where single-percent improvements compound across thousands of SKUs per week.

---

## **Executive Summary**

### Business Opportunities

* **Opportunity A:** A regional grocer running four stores across South-Central Texas faces weekly revenue variability driven by promotional pricing, shelf placement, and store-level demand patterns. Without per-product per-store revenue forecasts, inventory and promo decisions rely on rolling averages that miss store-specific and category-specific signal.

* **Solution A:** The trained regression ensemble produces per-product weekly revenue predictions at {{PASS_2: primary_r2}} R-squared, giving inventory planners a per-SKU forecast that incorporates store identity, promo state, shelf facings, and product attributes.

* **Opportunity B:** Perishable departments (Produce, Dairy, Meat, Bakery) carry the highest waste exposure when forecasts are wrong. Overstock on perishables becomes spoilage; understock becomes lost sales and customer dissatisfaction.

* **Solution B:** Department is a top-three feature in the trained model, allowing the forecast to weight perishable-specific patterns differently from shelf-stable categories. The model's MAPE of {{PASS_2: primary_mape}} translates to revenue prediction error within practical inventory tolerances.

* **Opportunity C:** Multiple deployment paths exist for this model across grocery operations and retail analytics workflows.

* **Solution C** (Options): Embedded forecasting tool inside the store-manager weekly planning dashboard. API integration with category-management software for promo planning. Standalone interactive forecasting interface for inventory analysts running what-if scenarios on price and promo combinations.

### Outcomes

* **Model Performance**

  * {{PASS_2: primary_model_name}} (tuned) achieves R-squared of {{PASS_2: primary_r2}}, MAPE of {{PASS_2: primary_mape}}, and RMSE of {{PASS_2: primary_rmse}} on the held-out test set.

  * {{PASS_2: secondary_model_name}} is the strongest challenger with R-squared of {{PASS_2: secondary_r2}}, retained for ensemble experimentation and deployment redundancy.

* **Architecture**

  * Six tabular regressors were compared under repeated cross-validation: Decision Tree, Bagging, Random Forest, Gradient Boosting, XGBoost, and CatBoost.

  * Top two models proceed to deployment as primary and secondary; the rest remain as benchmarks for future challenger development.

* **Economic Impact**

  * For a four-store grocer doing approximately {{PASS_2: total_weekly_revenue}} in weekly revenue across roughly {{PASS_2: total_rows}} product-store-weeks, a model that predicts to within {{PASS_2: primary_mape}} MAPE represents meaningful reduction in both spoilage and stockout cost.

  * Specific dollar impact depends on the customer's overstock and stockout cost structure, which are typically estimated during the engagement.

* **Strategy Recommendation**

  * **Enterprise:** Deploy as a forecasting microservice integrated with category-management and inventory-planning systems. Use challenger model in parallel for A/B comparison during the first two quarters of production use.

  * **SMB:** Deploy via the standalone web interface for store managers and inventory analysts. Single-prediction and batch-upload modes both supported, no internal systems integration required.

### Live Deployment

* **LOAD FIRST: Backend (API)**: https://huggingface.co/spaces/EvagAIML/ref-hill-country-grocer-revenue-pred-backend

* **LOAD SECOND: Frontend (UI)**: https://huggingface.co/spaces/EvagAIML/ref-hill-country-grocer-revenue-pred-frontend

---

## **Problem Space**

### Overview

* Hill Country Grocer is a regional independent grocer operating four stores across South-Central Texas. Weekly revenue per product-per-store is the core operational planning unit.

* Over-predicting revenue on perishables creates spoilage and waste. Under-predicting revenue on any product creates stockouts and lost sales.

* The data captures one full weekly product-store snapshot per row, with pricing, promo, shelf, and store-context features.

### Data Description

* {{PASS_2: total_rows}} product-store-week observations across four stores in South - Texas Central

* 16 columns: 15 raw features plus `Weekly_Revenue_USD` (target)

* Data file: `hill_country_grocer_weekly_sales.csv`

* Target: `Weekly_Revenue_USD` (continuous, USD)

* Train / validation / test split: {{PASS_2: train_rows}} / {{PASS_2: val_rows}} / {{PASS_2: test_rows}} (70/15/15 shuffled)

### Process

* Schema inspection identifies one target-leakage risk (`Weekly_Units_Sold` is mechanically a component of `Weekly_Revenue_USD` and must be dropped before modeling), one high-cardinality identifier (`UPC`), and several engineering opportunities (promo discount depth, store age).

* Six tabular regressors compared under 5-fold cross-validation on the training set, with GridSearchCV hyperparameter tuning per model.

* Top two models confirmed on held-out validation set; final selected model evaluated single-shot on held-out test set.

### Results

> | Model | R-squared (Test) | MAPE (Test) | RMSE (Test) | Current Role |
> |---|---|---|---|---|
> | **{{PASS_2: primary_model_name}} (Tuned) ✅** | **{{PASS_2: primary_r2}}** | **{{PASS_2: primary_mape}}** | **{{PASS_2: primary_rmse}}** | Lead deployment model |
> | {{PASS_2: secondary_model_name}} (Tuned) | {{PASS_2: secondary_r2}} | {{PASS_2: secondary_mape}} | {{PASS_2: secondary_rmse}} | Challenger / ensemble candidate |


# **Code Execution**


### **Runtime Configuration**

> **Hardware Accelerator:** **CPU** is sufficient (tree-based ensembles, modest dataset size).
>
> Set the Colab runtime to CPU. GPU is not required and provides no speedup for this workload. Ensure `HF_TOKEN` is set as a Colab secret (Runtime → Secrets) before executing the deployment cells.


### **Library Installation**

**Process:** Pinned dependency versions are installed to ensure reproducible model training, evaluation, and deployment-asset generation across notebook runs.

**Outcome:** All required libraries available at fixed versions; runtime restart required before subsequent cells execute.


In [ ]:
#-------------
# LIBRARY INSTALLATION
#-------------
# Pinned versions for reproducibility across runs and Colab environment refreshes.

!pip install -q numpy==2.0.2 pandas==2.2.2 scikit-learn==1.6.1 matplotlib==3.10.0 seaborn==0.13.2 joblib==1.4.2 xgboost==2.1.4 catboost==1.2.8 huggingface_hub==0.30.1 2>/dev/null

# Restart runtime banner so the kernel picks up the pinned versions.
from IPython.display import HTML
HTML("""
<div style='border: 2px solid #d33; padding: 12px; background: #fee; border-radius: 4px; font-family: sans-serif;'>
<strong style='color: #d33;'>⚠ Runtime Restart Required</strong><br>
Pinned libraries were installed. Use the Colab menu: <strong>Runtime → Restart session</strong>, then re-run all cells from the top.
</div>
""")


### **Library Import and Configuration**

**Process:** Tabular analysis, visualization, preprocessing, evaluation, and ensemble-regression libraries are loaded; display options are configured for the modeling workflow.

**Outcome:** Notebook environment is ready to load data, run EDA, build models, and generate deployment assets.


In [ ]:
#-------------
# LIBRARY IMPORT AND CONFIGURATION
#-------------
# Tabular data, visualization, preprocessing, ensemble regressors, and HuggingFace Hub client.

# 2.1 Standard library
import json  # for writing model_metadata.json during deployment asset generation
import os  # for environment-variable access (HF_TOKEN)
import shutil  # for managing deployment asset directories
import warnings  # for suppressing non-critical training warnings
from pathlib import Path  # for portable filesystem paths

# 2.2 Data and numerical computation
import joblib  # for serializing trained pipelines for deployment
import numpy as np  # for numerical operations and array handling
import pandas as pd  # for tabular data loading, cleaning, and feature engineering

# 2.3 Visualization
import matplotlib.pyplot as plt  # for univariate and bivariate plots
import seaborn as sns  # for statistical visualizations and heatmaps

# 2.4 Preprocessing, splitting, and evaluation
from sklearn.compose import ColumnTransformer  # for separate numeric and categorical pipelines
from sklearn.impute import SimpleImputer  # for missing-value imputation (fit on train only)
from sklearn.metrics import (  # for regression evaluation
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import (  # for train/val/test split, CV, and tuning
    GridSearchCV,
    KFold,
    RepeatedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline  # for chaining preprocessing and modeling (no leakage)
from sklearn.preprocessing import OneHotEncoder  # for categorical encoding

# 2.5 Regression model candidates
from sklearn.tree import DecisionTreeRegressor  # baseline interpretable tree
from sklearn.ensemble import (  # ensemble candidates
    BaggingRegressor,
    GradientBoostingRegressor,
    RandomForestRegressor,
)
from xgboost import XGBRegressor  # gradient-boosted trees, strong tabular default
from catboost import CatBoostRegressor  # gradient-boosted trees with native categorical handling

# 2.6 HuggingFace deployment client
from huggingface_hub import HfApi, create_repo, login, upload_folder  # for programmatic Space creation and push

# 2.7 Display and reproducibility configuration
pd.set_option("display.max_columns", None)  # show all columns when inspecting DataFrames
pd.set_option("display.max_rows", 100)  # cap row display to keep notebook output readable
warnings.filterwarnings("ignore")  # suppress non-critical warnings during training

RANDOM_STATE = 1  # reproducibility seed used across splits, CV folds, and stochastic estimators

print("Libraries loaded.")


### **Data Loading**

**Process:** The Hill Country Grocer weekly sales dataset is loaded from the public reference-data repo via the slug-based raw URL pattern. The raw DataFrame is preserved untouched; all downstream work operates on a working copy.

**Outcome:** Source dataset is available as `raw_df`; `data` holds the working copy that subsequent cells modify.


In [ ]:
#-------------
# DATA LOADING
#-------------
# Load the engagement's raw CSV from the public reference-data repo; preserve raw, work on a copy.

ENGAGEMENT_SLUG = "ref-hill-country-grocer-revenue-pred__reg__ensemble-exp"
DATA_REPO_BASE = "https://raw.githubusercontent.com/EvagAIML/000-smb-consulting-reference-data/main"
DATASET_PATH = f"{DATA_REPO_BASE}/engagements/{ENGAGEMENT_SLUG}/data/raw/hill_country_grocer_weekly_sales.csv"

raw_df = pd.read_csv(DATASET_PATH)  # raw is immutable; never modified after load
data = raw_df.copy()  # working copy for all cleaning and feature engineering

print(f"Loaded {len(data):,} rows and {len(data.columns)} columns from the engagement's data/raw/ path.")


### **Data Overview**

**Process:** The working dataset is inspected for shape, dtypes, missing values, duplicates, and first-row content. This establishes the schema baseline before any feature decisions.

**Analysis:** {{PASS_2: data_overview_analysis}}

**Outcome:** Schema, missingness, and duplicate counts are confirmed; downstream feature decisions reference these results.


In [ ]:
#-------------
# DATA OVERVIEW
#-------------
# Shape, dtypes, missingness, duplicates, head — baseline schema understanding before any drops.

# 4.1 Shape
print(f"Rows: {data.shape[0]:,}")
print(f"Columns: {data.shape[1]}")

# 4.2 Dtypes and non-null counts
print("\nSchema and dtypes:")
data.info()

# 4.3 Missing-value totals per column
print("\nMissing values per column:")
print(data.isnull().sum())

# 4.4 Duplicate rows
print(f"\nDuplicate rows: {data.duplicated().sum()}")

# 4.5 First five rows for content sanity check
print("\nFirst five rows:")
data.head()


### **Target and Feature Decisions**

**Process:** Before any modeling, three feature decisions are made based on the schema inspection above. Each decision is documented in code with reasoning.

**Analysis:** Three risks need to be addressed up front. (1) `Weekly_Units_Sold` is a mechanical component of the target `Weekly_Revenue_USD` (revenue ≈ units × price), so keeping it as a feature would create target leakage and produce inflated metrics that do not generalize to a real forecasting setting where units sold is unknown at prediction time. (2) `UPC` is a unique product identifier with thousands of distinct values; one-hot encoding it would explode the feature space without providing reusable signal for new products, so it must be dropped. (3) `Item_Description` is high-cardinality text that is largely redundant with `Department`, which already provides a clean categorical signal at the right granularity for a tabular regression model.

**Outcome:** The feature set is reduced from 15 raw features to 12 modeling features with one target column.


In [ ]:
#-------------
# TARGET AND FEATURE DECISIONS
#-------------
# Document each drop with a one-line reason so the decision is auditable in code review.

TARGET = "Weekly_Revenue_USD"

# 5.1 Drop columns that would harm modeling
columns_to_drop = [
    "Weekly_Units_Sold",   # target leakage: mechanically determines revenue with price
    "UPC",                  # high-cardinality identifier, no reusable signal
    "Item_Description",     # redundant with Department, high-cardinality text
]

data = data.drop(columns=columns_to_drop)

print("Dropped columns:")
for col in columns_to_drop:
    print(f"  - {col}")
print(f"\nRemaining columns: {len(data.columns)} ({len(data.columns) - 1} features + 1 target)")
print(list(data.columns))


### **Feature Engineering**

**Process:** Two engineered features are added that capture business-relevant signal not directly present in the raw columns: `Store_Age_Years` summarizes store maturity more usefully than the establishment year, and `Promo_Discount_Pct` captures the depth of promotional pricing as a ratio rather than relying on the model to learn the relationship between `List_Price_USD` and `Promo_Price_USD` from scratch.

**Outcome:** Two new features added; `Store_Open_Year` replaced by `Store_Age_Years`. Final feature count is {{PASS_2: final_feature_count}}.


In [ ]:
#-------------
# FEATURE ENGINEERING
#-------------
# Add business-relevant derived features; remove the now-redundant raw year.

CURRENT_YEAR = 2026

# 5.2 Store_Age_Years from Store_Open_Year
data["Store_Age_Years"] = CURRENT_YEAR - data["Store_Open_Year"]
data = data.drop(columns=["Store_Open_Year"])

# 5.3 Promo_Discount_Pct from list and promo prices
data["Promo_Discount_Pct"] = (
    (data["List_Price_USD"] - data["Promo_Price_USD"]) / data["List_Price_USD"]
).clip(lower=0)  # negative discount would indicate a data error; clip to zero

print("Engineered features added:")
print(f"  - Store_Age_Years (replaces Store_Open_Year)")
print(f"  - Promo_Discount_Pct")
print(f"\nFinal feature count: {len(data.columns) - 1} features + 1 target")
print(list(data.columns))


### **Univariate Distributions: Target and Key Numeric Features**

**Process:** Distributions of the target and the most operationally important numeric features are visualized to understand scale, skew, and the presence of outliers.

**Analysis:** {{PASS_2: univariate_analysis}}

**Outcome:** Distribution shape for `Weekly_Revenue_USD`, `List_Price_USD`, and `Promo_Discount_Pct` is established as input to model selection (e.g., tree-based ensembles handle skew without scaling).


In [ ]:
#-------------
# UNIVARIATE DISTRIBUTIONS
#-------------
numeric_features_to_plot = ["Weekly_Revenue_USD", "List_Price_USD", "Promo_Discount_Pct", "Shelf_Facings"]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, col in zip(axes.flat, numeric_features_to_plot):
    sns.histplot(data=data, x=col, kde=True, ax=ax)
    ax.set_title(f"Distribution: {col}")
plt.tight_layout()
plt.show()

print("\nSummary statistics for plotted features:")
data[numeric_features_to_plot].describe()


### **Categorical Frequencies: Department, Store, Brand**

**Process:** Category frequencies are inspected for `Department`, `Store_Number`, and `Brand_Type` to confirm class balance and identify any rare categories.

**Analysis:** {{PASS_2: categorical_analysis}}

**Outcome:** Category-level frequencies are confirmed; rare categories (if any) are flagged for encoding strategy.


In [ ]:
#-------------
# CATEGORICAL FREQUENCIES
#-------------
categorical_features_to_plot = ["Department", "Store_Number", "Brand_Type"]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, col in zip(axes, categorical_features_to_plot):
    counts = data[col].value_counts()
    sns.barplot(x=counts.index, y=counts.values, ax=ax)
    ax.set_title(f"Frequency: {col}")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

for col in categorical_features_to_plot:
    print(f"\n{col} value counts:")
    print(data[col].value_counts())


### **Bivariate: Drivers of Weekly Revenue**

**Process:** Relationships between the target and the most likely driver features are visualized: promo price, list price, promo discount depth, and per-store revenue distribution.

**Analysis:** {{PASS_2: bivariate_analysis}}

**Outcome:** Direction and strength of key drivers are established; non-linear relationships (if any) inform the choice of tree-based models over linear baselines.


In [ ]:
#-------------
# BIVARIATE: REVENUE DRIVERS
#-------------
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.scatterplot(data=data, x="Promo_Price_USD", y="Weekly_Revenue_USD", ax=axes[0, 0], alpha=0.3)
axes[0, 0].set_title("Promo Price vs Weekly Revenue")

sns.scatterplot(data=data, x="List_Price_USD", y="Weekly_Revenue_USD", ax=axes[0, 1], alpha=0.3)
axes[0, 1].set_title("List Price vs Weekly Revenue")

sns.scatterplot(data=data, x="Promo_Discount_Pct", y="Weekly_Revenue_USD", ax=axes[1, 0], alpha=0.3)
axes[1, 0].set_title("Promo Discount % vs Weekly Revenue")

sns.boxplot(data=data, x="Store_Number", y="Weekly_Revenue_USD", ax=axes[1, 1])
axes[1, 1].set_title("Revenue Distribution by Store")
axes[1, 1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

numeric_cols = data.select_dtypes(include=np.number).columns.tolist()
plt.figure(figsize=(10, 8))
sns.heatmap(data[numeric_cols].corr(), annot=True, fmt=".2f", cmap="Spectral", vmin=-1, vmax=1)
plt.title("Numeric Feature Correlation Matrix")
plt.tight_layout()
plt.show()


### **Data Preparation for Modeling**

**Process:** Features and target are separated, then a 70/15/15 train/validation/test split is created using a fixed random seed. The split happens before any preprocessing so imputation, scaling, and encoding are fit on the training set only.

**Outcome:** Three disjoint sets are produced: training ({{PASS_2: train_rows}} rows) for model fitting, validation ({{PASS_2: val_rows}} rows) for model selection and hyperparameter tuning, and test ({{PASS_2: test_rows}} rows) reserved for single-shot final evaluation.


In [ ]:
#-------------
# DATA PREPARATION FOR MODELING
#-------------
X = data.drop(columns=[TARGET])
y = data[TARGET]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, shuffle=True,
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, shuffle=True,
)

print(f"Train: {X_train.shape}")
print(f"Val:   {X_val.shape}")
print(f"Test:  {X_test.shape}")
print(f"\nTarget distribution:")
print(f"  Train mean: {y_train.mean():.2f}, std: {y_train.std():.2f}")
print(f"  Val mean:   {y_val.mean():.2f}, std: {y_val.std():.2f}")
print(f"  Test mean:  {y_test.mean():.2f}, std: {y_test.std():.2f}")


### **Preprocessing Pipeline**

**Process:** A `ColumnTransformer` is defined that imputes missing numeric values with the median, imputes missing categoricals with the most-frequent value, and one-hot-encodes categorical columns. The transformer is fit on training data only when used inside a `Pipeline`; this is the canonical way to prevent test-set leakage during preprocessing.

**Outcome:** A reusable `preprocessor` object is defined that will be chained with each model candidate in the next step.


In [ ]:
#-------------
# PREPROCESSING PIPELINE
#-------------
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

numeric_transformer = Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

print(f"Numeric features: {numeric_features}")
print(f"Categorical features: {categorical_features}")


### **Metric and Evaluation Utilities**

**Process:** Helper functions compute the standard regression metric set and run the full evaluation pipeline (CV + tuning + final fit) for a single model candidate. Centralizing this logic ensures every model is evaluated identically.

**Outcome:** `get_metrics` and `evaluate_model` are defined; subsequent model cells call them rather than re-implementing the evaluation logic.


In [ ]:
#-------------
# METRIC AND EVALUATION UTILITIES
#-------------
VALIDATION_CV = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
TUNING_CV = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
VALIDATION_SCORING = {
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "r2": "r2",
    "mape": "neg_mean_absolute_percentage_error",
}

def get_metrics(y_true, y_pred, subset_label):
    return {
        f"{subset_label}_RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        f"{subset_label}_MAE": mean_absolute_error(y_true, y_pred),
        f"{subset_label}_R2": r2_score(y_true, y_pred),
        f"{subset_label}_MAPE": mean_absolute_percentage_error(y_true, y_pred),
    }

def evaluate_model(model_name, model, param_grid):
    print(f"Evaluating {model_name}...")
    pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("regressor", model)])
    pipeline.fit(X_train, y_train)
    base_val_pred = pipeline.predict(X_val)
    base_metrics = get_metrics(y_val, base_val_pred, "BaseVal")

    search = GridSearchCV(pipeline, param_grid, cv=TUNING_CV,
                         scoring="neg_root_mean_squared_error", n_jobs=-1)
    search.fit(X_train, y_train)
    tuned_pipeline = search.best_estimator_

    cv_scores = cross_validate(tuned_pipeline, X_train, y_train, cv=VALIDATION_CV,
                               scoring=VALIDATION_SCORING, n_jobs=1, return_train_score=False)
    cv_summary = {
        "CV_RMSE_Mean": -cv_scores["test_rmse"].mean(),
        "CV_RMSE_Std": cv_scores["test_rmse"].std(),
        "CV_MAE_Mean": -cv_scores["test_mae"].mean(),
        "CV_R2_Mean": cv_scores["test_r2"].mean(),
        "CV_MAPE_Mean": -cv_scores["test_mape"].mean(),
    }

    tuned_val_pred = tuned_pipeline.predict(X_val)
    tuned_val_metrics = get_metrics(y_val, tuned_val_pred, "TunedVal")

    print(f"  Best params: {search.best_params_}")
    print(f"  CV RMSE mean: {cv_summary['CV_RMSE_Mean']:.4f}")
    print(f"  Val R2 (tuned): {tuned_val_metrics['TunedVal_R2']:.4f}")

    return {
        "base_val_metrics": base_metrics,
        "tuned_val_metrics": tuned_val_metrics,
        "cv_summary": cv_summary,
        "best_params": search.best_params_,
        "fitted_pipeline": tuned_pipeline,
    }


### **Model Candidates**

**Process:** Six tabular regressors are defined with hyperparameter grids appropriate to each model's complexity. Decision Tree serves as the interpretable baseline; Bagging, Random Forest, Gradient Boosting, XGBoost, and CatBoost are the ensemble candidates.

**Outcome:** A `models` dict and `param_grids` dict define the full benchmarking lineup.


In [ ]:
#-------------
# MODEL CANDIDATES
#-------------
models = {
    "Decision Tree": DecisionTreeRegressor(random_state=RANDOM_STATE),
    "Bagging": BaggingRegressor(random_state=RANDOM_STATE, n_jobs=1),
    "Random Forest": RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
    "XGBoost": XGBRegressor(random_state=RANDOM_STATE, objective="reg:squarederror", verbosity=0, n_jobs=1),
    "CatBoost": CatBoostRegressor(random_state=RANDOM_STATE, verbose=0, allow_writing_files=False, thread_count=1),
}

param_grids = {
    "Decision Tree": {"regressor__max_depth": [3, 5, 7, None], "regressor__min_samples_leaf": [1, 5, 10]},
    "Bagging": {"regressor__n_estimators": [50, 100], "regressor__max_samples": [0.7, 1.0]},
    "Random Forest": {"regressor__n_estimators": [100], "regressor__max_depth": [10, None], "regressor__max_features": ["sqrt", None]},
    "Gradient Boosting": {"regressor__n_estimators": [100, 200], "regressor__learning_rate": [0.05, 0.1], "regressor__max_depth": [3, 5]},
    "XGBoost": {"regressor__n_estimators": [100, 200], "regressor__learning_rate": [0.05, 0.1], "regressor__max_depth": [3, 5], "regressor__subsample": [0.8], "regressor__colsample_bytree": [0.8]},
    "CatBoost": {"regressor__iterations": [300, 500], "regressor__learning_rate": [0.05, 0.1], "regressor__depth": [4, 6, 8]},
}

print(f"Defined {len(models)} model candidates with tuning grids.")


### **Training and Evaluation Loop**

**Process:** Each model is evaluated through the shared `evaluate_model` utility: fit-on-train, grid-search-tune, 5-fold cross-validate, and score on the validation set.

**Outcome:** The `results` dict holds tuned pipelines and metric summaries for every candidate.


In [ ]:
#-------------
# TRAINING AND EVALUATION LOOP
#-------------
results = {}
for model_name, model in models.items():
    results[model_name] = evaluate_model(model_name, model, param_grids[model_name])

print("\nAll candidates evaluated.")


### **Model Comparison Summary**

**Process:** Validation-set metrics for every tuned model are assembled into a single comparison DataFrame and ranked by validation RMSE.

**Analysis:** {{PASS_2: comparison_analysis}}

**Outcome:** The ranked DataFrame identifies the top two candidates that will proceed to test-set evaluation and deployment.


In [ ]:
#-------------
# MODEL COMPARISON SUMMARY
#-------------
comparison_rows = []
for model_name, res in results.items():
    comparison_rows.append({
        "Model": model_name,
        "Val_RMSE": res["tuned_val_metrics"]["TunedVal_RMSE"],
        "Val_MAE": res["tuned_val_metrics"]["TunedVal_MAE"],
        "Val_R2": res["tuned_val_metrics"]["TunedVal_R2"],
        "Val_MAPE": res["tuned_val_metrics"]["TunedVal_MAPE"],
        "CV_RMSE_Mean": res["cv_summary"]["CV_RMSE_Mean"],
        "CV_RMSE_Std": res["cv_summary"]["CV_RMSE_Std"],
    })

comparison_df = pd.DataFrame(comparison_rows).set_index("Model")
comparison_df = comparison_df.sort_values(by=["Val_RMSE", "Val_MAE", "Val_R2"], ascending=[True, True, False])

print("Model comparison (ranked by validation RMSE):")
comparison_df


In [ ]:
#-------------
# TOP TWO SELECTION
#-------------
ranked_models = comparison_df.index.tolist()
primary_model_name = ranked_models[0]
secondary_model_name = ranked_models[1]

primary_pipeline = results[primary_model_name]["fitted_pipeline"]
secondary_pipeline = results[secondary_model_name]["fitted_pipeline"]

print(f"Primary model:   {primary_model_name}")
print(f"Secondary model: {secondary_model_name}")
print(f"\nPrimary best params:   {results[primary_model_name]['best_params']}")
print(f"Secondary best params: {results[secondary_model_name]['best_params']}")


### **Test Set Evaluation**

**Process:** The primary and secondary models are evaluated once on the held-out test set. This is a single-shot evaluation.

**Analysis:** {{PASS_2: test_evaluation_analysis}}

**Outcome:** Final generalization metrics confirm (or revise) the model selection from the validation-based ranking.


In [ ]:
#-------------
# TEST SET EVALUATION
#-------------
primary_test_pred = primary_pipeline.predict(X_test)
secondary_test_pred = secondary_pipeline.predict(X_test)

primary_test_metrics = get_metrics(y_test, primary_test_pred, "Test")
secondary_test_metrics = get_metrics(y_test, secondary_test_pred, "Test")

test_summary = pd.DataFrame({
    primary_model_name: primary_test_metrics,
    secondary_model_name: secondary_test_metrics,
}).T

print("Held-out test set performance:")
test_summary


### **Business Alignment**

**Process:** Test-set metrics are translated into business-meaningful terms.

**Analysis:** {{PASS_2: business_alignment_analysis}}

**Outcome:** A plain-language summary of what the model's accuracy means for weekly inventory and promo decisions.


In [ ]:
#-------------
# BUSINESS ALIGNMENT
#-------------
total_test_revenue = y_test.sum()
mean_test_revenue = y_test.mean()
primary_mape_pct = primary_test_metrics["Test_MAPE"] * 100
primary_rmse_usd = primary_test_metrics["Test_RMSE"]

print(f"Total test-set revenue:  ${total_test_revenue:,.2f}")
print(f"Mean per-row revenue:    ${mean_test_revenue:,.2f}")
print(f"Primary model MAPE:      {primary_mape_pct:.2f}%")
print(f"Primary model RMSE:      ${primary_rmse_usd:,.2f}")
print(f"\nInterpretation: on average, the primary model predicts weekly revenue per product-store within {primary_mape_pct:.1f}% of actual,")
print(f"with typical error magnitude (RMSE) of ${primary_rmse_usd:,.2f} per row.")


---

# **Deployment**

The following cells produce deployment assets for the primary and secondary models, create the two HuggingFace Spaces programmatically, and push assets to both.

**Prerequisite:** `HF_TOKEN` must be set as a Colab secret before running these cells.


### **Model Serialization**

**Process:** Primary and secondary pipelines are saved as `.joblib` files. A `model_metadata.json` records model identities, display labels, and holdout test metrics for the backend to read at load time.

**Outcome:** Three artifacts in `deployment_files/`.


In [ ]:
#-------------
# MODEL SERIALIZATION
#-------------
deployment_dir = Path("deployment_files")
if deployment_dir.exists():
    shutil.rmtree(deployment_dir)
deployment_dir.mkdir()

PRIMARY_ARTIFACT = deployment_dir / "primary_model.joblib"
SECONDARY_ARTIFACT = deployment_dir / "secondary_model.joblib"
METADATA_PATH = deployment_dir / "model_metadata.json"

joblib.dump(primary_pipeline, PRIMARY_ARTIFACT)
joblib.dump(secondary_pipeline, SECONDARY_ARTIFACT)

metadata = {
    "primary_model_name": primary_model_name,
    "secondary_model_name": secondary_model_name,
    "primary_label": primary_model_name,
    "secondary_label": secondary_model_name,
    "primary_holdout_metrics": {k: float(v) for k, v in primary_test_metrics.items()},
    "secondary_holdout_metrics": {k: float(v) for k, v in secondary_test_metrics.items()},
}
METADATA_PATH.write_text(json.dumps(metadata, indent=2))

print("Serialized artifacts:")
for path in [PRIMARY_ARTIFACT, SECONDARY_ARTIFACT, METADATA_PATH]:
    print(f"  - {path}")


### **Backend Asset Generation**

**Process:** A `backend_space_root/` directory is assembled containing the serialized models, metadata, a Dockerfile, a pinned `requirements.txt`, a FastAPI `app.py`, and a README with HF Space YAML frontmatter.

**Outcome:** The backend Space is fully prepared as a single directory ready to push.


In [ ]:
#-------------
# BACKEND ASSET GENERATION
#-------------
backend_root = Path("backend_space_root")
if backend_root.exists():
    shutil.rmtree(backend_root)
backend_root.mkdir()

shutil.copy(PRIMARY_ARTIFACT, backend_root / "primary_model.joblib")
shutil.copy(SECONDARY_ARTIFACT, backend_root / "secondary_model.joblib")
shutil.copy(METADATA_PATH, backend_root / "model_metadata.json")

readme_backend = """---
title: ref-hill-country-grocer-revenue-pred-backend
emoji: 🛒
colorFrom: blue
colorTo: green
sdk: docker
app_port: 7860
pinned: false
---

# Hill Country Grocer Revenue Prediction — Backend

FastAPI prediction service. Endpoints: `GET /`, `GET /health`, `POST /v1/predict`.
"""
(backend_root / "README.md").write_text(readme_backend)

dockerfile_backend = """FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 7860
CMD [\"uvicorn\", \"app:app\", \"--host\", \"0.0.0.0\", \"--port\", \"7860\"]
"""
(backend_root / "Dockerfile").write_text(dockerfile_backend)

requirements_backend = """fastapi==0.111.0
uvicorn==0.30.1
scikit-learn==1.6.1
joblib==1.4.2
pandas==2.2.2
numpy==2.0.2
xgboost==2.1.4
catboost==1.2.8
"""
(backend_root / "requirements.txt").write_text(requirements_backend)

app_backend = '''from fastapi import FastAPI, HTTPException
from typing import Any, Dict
import json
import joblib
import pandas as pd

app = FastAPI()

with open("model_metadata.json", "r") as fh:
    metadata = json.load(fh)

PRIMARY_LABEL = metadata["primary_label"]
SECONDARY_LABEL = metadata["secondary_label"]

MODELS = {
    PRIMARY_LABEL: joblib.load("primary_model.joblib"),
    SECONDARY_LABEL: joblib.load("secondary_model.joblib"),
}

@app.get("/")
def root():
    return {
        "status": "healthy",
        "message": "Hill Country Grocer Revenue Prediction backend. Use /v1/predict.",
        "primary_model": PRIMARY_LABEL,
        "secondary_model": SECONDARY_LABEL,
    }

@app.get("/health")
def health():
    return {"status": "healthy", "primary_model": PRIMARY_LABEL, "secondary_model": SECONDARY_LABEL}

@app.post("/v1/predict")
async def predict(request_body: Dict[str, Any]):
    model_name = request_body.get("model", PRIMARY_LABEL)
    rows = request_body.get("rows", [])
    if not rows:
        raise HTTPException(status_code=400, detail="No rows provided.")
    model = MODELS.get(model_name)
    if model is None:
        raise HTTPException(status_code=404, detail=f"Model not found: {model_name}")
    try:
        df = pd.DataFrame(rows)
        preds = model.predict(df).tolist()
        overall_total = float(sum(preds))
        store_totals = {}
        if "Store_Number" in df.columns:
            tmp = df.copy()
            tmp["prediction"] = preds
            store_totals = {k: float(v) for k, v in tmp.groupby("Store_Number")["prediction"].sum().to_dict().items()}
        return {
            "model_used": model_name,
            "predictions": preds,
            "overall_total": overall_total,
            "store_totals": store_totals,
        }
    except Exception as exc:
        raise HTTPException(status_code=500, detail=str(exc))
'''
(backend_root / "app.py").write_text(app_backend)

print("Backend assets staged:")
for path in sorted(backend_root.iterdir()):
    print(f"  - {path.name}")


### **Backend Validation**

**Process:** Required files in `backend_space_root/` are confirmed present and YAML frontmatter is verified.

**Outcome:** All assertions pass; the backend directory is ready to push.


In [ ]:
#-------------
# BACKEND VALIDATION
#-------------
required_backend_files = ["primary_model.joblib", "secondary_model.joblib", "model_metadata.json", "Dockerfile", "requirements.txt", "app.py", "README.md"]
for fname in required_backend_files:
    assert (backend_root / fname).exists(), f"Missing backend file: {fname}"

readme_text = (backend_root / "README.md").read_text()
assert "sdk: docker" in readme_text
assert "app_port: 7860" in readme_text

print("Backend validation PASSED.")


### **HuggingFace Login**

**Process:** The HF_TOKEN Colab secret authenticates the `huggingface_hub` client.

**Outcome:** Authenticated session.


In [ ]:
#-------------
# HUGGINGFACE LOGIN
#-------------
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except (ImportError, Exception):
    hf_token = os.environ.get("HF_TOKEN")

if not hf_token:
    raise RuntimeError("HF_TOKEN not found. Set it as a Colab secret before running this cell.")

login(token=hf_token)
print("Authenticated with HuggingFace.")


### **Programmatic Space Creation**

**Process:** Both Spaces are created via `create_repo` with `space_sdk="docker"`. `exist_ok=True` makes re-runs idempotent.

**Outcome:** Both Spaces exist on HuggingFace under the `EvagAIML` namespace.


In [ ]:
#-------------
# PROGRAMMATIC SPACE CREATION
#-------------
SHORT_SLUG = "ref-hill-country-grocer-revenue-pred"
BACKEND_REPO_ID = f"EvagAIML/{SHORT_SLUG}-backend"
FRONTEND_REPO_ID = f"EvagAIML/{SHORT_SLUG}-frontend"

create_repo(repo_id=BACKEND_REPO_ID, repo_type="space", space_sdk="docker", private=False, exist_ok=True)
print(f"Backend Space ready: {BACKEND_REPO_ID}")

create_repo(repo_id=FRONTEND_REPO_ID, repo_type="space", space_sdk="docker", private=False, exist_ok=True)
print(f"Frontend Space ready: {FRONTEND_REPO_ID}")


### **Backend Push**

**Process:** Backend assets uploaded to the backend Space; HF rebuilds the container.

**Outcome:** Backend Space goes live at the URL declared above.


In [ ]:
#-------------
# BACKEND PUSH
#-------------
upload_folder(
    repo_id=BACKEND_REPO_ID,
    folder_path=str(backend_root),
    path_in_repo=".",
    repo_type="space",
    commit_message="Pass 1 deploy: backend (FastAPI + Docker)",
    create_pr=False,
)
print(f"Backend pushed to https://huggingface.co/spaces/{BACKEND_REPO_ID}")


### **Frontend Asset Generation**

**Process:** A `frontend_space_root/` directory is assembled with Streamlit-in-Docker setup.

**Outcome:** Frontend Space is fully prepared as a single directory ready to push.


In [ ]:
#-------------
# FRONTEND ASSET GENERATION
#-------------
frontend_root = Path("frontend_space_root")
if frontend_root.exists():
    shutil.rmtree(frontend_root)
frontend_root.mkdir()
(frontend_root / "src").mkdir()

readme_frontend = """---
title: ref-hill-country-grocer-revenue-pred-frontend
emoji: 🛒
colorFrom: green
colorTo: blue
sdk: docker
app_port: 8501
pinned: false
---

# Hill Country Grocer Revenue Prediction — Frontend

Streamlit-in-Docker UI that calls the backend prediction API.
"""
(frontend_root / "README.md").write_text(readme_frontend)

dockerfile_frontend = """FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 8501
CMD [\"streamlit\", \"run\", \"src/streamlit_app.py\", \"--server.port=8501\", \"--server.address=0.0.0.0\"]
"""
(frontend_root / "Dockerfile").write_text(dockerfile_frontend)

requirements_frontend = """streamlit==1.43.2
requests==2.32.3
"""
(frontend_root / "requirements.txt").write_text(requirements_frontend)

BACKEND_URL = f"https://EvagAIML-{SHORT_SLUG}-backend.hf.space/v1/predict"
PRIMARY_LABEL = primary_model_name
SECONDARY_LABEL = secondary_model_name

app_frontend = '''import csv
import io
from typing import Any, Dict, List

import requests
import streamlit as st

BACKEND_URL = "__BACKEND_URL__"
PRIMARY_LABEL = "__PRIMARY_LABEL__"
SECONDARY_LABEL = "__SECONDARY_LABEL__"
MODEL_OPTIONS = [PRIMARY_LABEL, SECONDARY_LABEL]

DEFAULTS = {
    "Store_Number": "NEW_STORE",
    "Store_Banner": "Hill Country Grocer",
    "Store_Region": "South - Texas Central",
    "Store_Sq_Ft": 35000,
    "Store_Age_Years": 10,
    "Department": "Produce",
    "Brand_Type": "National Brand",
    "Net_Weight_oz": 5.0,
    "Pack_Size": 1,
    "List_Price_USD": 2.50,
    "Promo_Price_USD": 2.50,
    "Shelf_Facings": 4,
}

STORE_OPTIONS = ["HCG-101", "HCG-102", "HCG-103", "HCG-104", "NEW_STORE"]
DEPARTMENT_OPTIONS = ["Produce", "Dairy", "Meat", "Bakery", "Frozen", "Beverages", "Snacks", "Pantry", "Health & Beauty", "Household"]
BRAND_TYPE_OPTIONS = ["National Brand", "Private Label", "Local Brand"]

def derive_promo_discount(list_price, promo_price):
    if list_price <= 0:
        return 0.0
    return max(0.0, (list_price - promo_price) / list_price)

def call_backend(model_name, rows):
    payload = {"model": model_name, "rows": rows}
    r = requests.post(BACKEND_URL, json=payload, timeout=60)
    r.raise_for_status()
    return r.json()

st.set_page_config(page_title="Hill Country Grocer Revenue Forecast", layout="centered")
st.title("Hill Country Grocer — Weekly Revenue Forecast")
st.caption("Predict weekly per-product per-store revenue.")

st.divider()
st.subheader("Administration")
model_choice = st.selectbox("Forecasting Model", MODEL_OPTIONS, index=0)
show_raw = st.checkbox("Show raw JSON response", value=False)

st.divider()
st.subheader("Single Prediction")

col1, col2 = st.columns(2)
with col1:
    store_number = st.selectbox("Store Number", STORE_OPTIONS, index=STORE_OPTIONS.index(DEFAULTS["Store_Number"]))
    store_banner = st.text_input("Store Banner", DEFAULTS["Store_Banner"])
    store_region = st.text_input("Store Region", DEFAULTS["Store_Region"])
    store_sq_ft = st.number_input("Store Sq Ft", min_value=1000, value=DEFAULTS["Store_Sq_Ft"])
    store_age = st.number_input("Store Age (years)", min_value=0, value=DEFAULTS["Store_Age_Years"])
with col2:
    department = st.selectbox("Department", DEPARTMENT_OPTIONS, index=DEPARTMENT_OPTIONS.index(DEFAULTS["Department"]))
    brand_type = st.selectbox("Brand Type", BRAND_TYPE_OPTIONS, index=BRAND_TYPE_OPTIONS.index(DEFAULTS["Brand_Type"]))
    net_weight = st.number_input("Net Weight (oz)", min_value=0.1, value=DEFAULTS["Net_Weight_oz"])
    pack_size = st.number_input("Pack Size", min_value=1, value=DEFAULTS["Pack_Size"])
    list_price = st.number_input("List Price (USD)", min_value=0.10, value=DEFAULTS["List_Price_USD"])
    promo_price = st.number_input("Promo Price (USD)", min_value=0.10, value=DEFAULTS["Promo_Price_USD"])
    shelf_facings = st.number_input("Shelf Facings", min_value=1, value=DEFAULTS["Shelf_Facings"])

promo_discount_pct = derive_promo_discount(list_price, promo_price)
st.caption(f"Derived Promo Discount %: {promo_discount_pct:.1%}")

row = {
    "Department": department,
    "Brand_Type": brand_type,
    "Net_Weight_oz": net_weight,
    "Pack_Size": pack_size,
    "List_Price_USD": list_price,
    "Promo_Price_USD": promo_price,
    "Shelf_Facings": shelf_facings,
    "Store_Number": store_number,
    "Store_Banner": store_banner,
    "Store_Region": store_region,
    "Store_Sq_Ft": store_sq_ft,
    "Store_Age_Years": store_age,
    "Promo_Discount_Pct": promo_discount_pct,
}

if st.button("Predict Revenue", type="primary"):
    try:
        result = call_backend(model_choice, [row])
        preds = result.get("predictions", [])
        if preds:
            st.metric("Predicted Weekly Revenue", f"${float(preds[0]):,.2f}")
        if show_raw:
            st.json(result)
    except Exception as exc:
        st.error(f"Prediction error: {exc}")
'''

app_frontend = app_frontend.replace("__BACKEND_URL__", BACKEND_URL)
app_frontend = app_frontend.replace("__PRIMARY_LABEL__", PRIMARY_LABEL)
app_frontend = app_frontend.replace("__SECONDARY_LABEL__", SECONDARY_LABEL)

(frontend_root / "src" / "streamlit_app.py").write_text(app_frontend)

print("Frontend assets staged.")


### **Frontend Validation**

**Process:** Required files confirmed, YAML frontmatter verified, placeholders substituted.

**Outcome:** All assertions pass; frontend ready to push.


In [ ]:
#-------------
# FRONTEND VALIDATION
#-------------
required_frontend_files = ["README.md", "Dockerfile", "requirements.txt", "src/streamlit_app.py"]
for fname in required_frontend_files:
    assert (frontend_root / fname).exists(), f"Missing frontend file: {fname}"

readme_text = (frontend_root / "README.md").read_text()
assert "sdk: docker" in readme_text
assert "app_port: 8501" in readme_text

reqs = (frontend_root / "requirements.txt").read_text()
assert "streamlit" in reqs
assert "pandas" not in reqs

app_text = (frontend_root / "src" / "streamlit_app.py").read_text()
assert "__BACKEND_URL__" not in app_text
assert "__PRIMARY_LABEL__" not in app_text
assert primary_model_name in app_text

print("Frontend validation PASSED.")


### **Frontend Push**

**Process:** Frontend assets uploaded; HF rebuilds container.

**Outcome:** Frontend Space live and calling the backend.


In [ ]:
#-------------
# FRONTEND PUSH
#-------------
upload_folder(
    repo_id=FRONTEND_REPO_ID,
    folder_path=str(frontend_root),
    path_in_repo=".",
    repo_type="space",
    commit_message="Pass 1 deploy: frontend (Streamlit-in-Docker)",
    create_pr=False,
)
print(f"Frontend pushed to https://huggingface.co/spaces/{FRONTEND_REPO_ID}")
print("\nDeployment complete. Both Spaces will rebuild on HF. First build may take several minutes.")


---

### **Expanded Executive Summary**

**TLDR**

Six tabular regressors were benchmarked for predicting Hill Country Grocer weekly product-store revenue across approximately 8,880 observations from four stores. The primary model, {{PASS_2: primary_model_name}}, achieves R-squared of {{PASS_2: primary_r2}} and MAPE of {{PASS_2: primary_mape}} on the held-out test set; {{PASS_2: secondary_model_name}} is retained as the challenger and deployment redundancy.

**Full Summary**

**Objective:** Produce per-product per-store weekly revenue forecasts for a four-store regional independent grocer, supporting inventory and promo decisions. Forecast accuracy directly affects spoilage on perishables and stockout losses.

**Data and Preparation:** {{PASS_2: total_rows}} product-store-week observations across four stores (HCG-101 through HCG-104) in South-Central Texas. Three columns dropped before modeling: `Weekly_Units_Sold` (target leakage), `UPC` (high-cardinality identifier), `Item_Description` (redundant with `Department`). Two engineered features added: `Store_Age_Years` and `Promo_Discount_Pct`. 70/15/15 train/validation/test split with shared random seed.

**Iterative Development:** Six candidates trained: Decision Tree, Bagging, Random Forest, Gradient Boosting, XGBoost, CatBoost. Each tuned via GridSearchCV with 3-fold CV, then 5-fold CV for model ranking.

**Performance Analysis:** {{PASS_2: performance_analysis}}

**Economic Impact:** {{PASS_2: economic_impact}}

**Deployment Readiness:** Both top models serialized as joblib pipelines and deployed to HuggingFace Spaces under the Docker SDK: FastAPI backend and Streamlit-in-Docker frontend. Both Spaces created programmatically.

**Next Steps:** {{PASS_2: next_steps}}
